# HemaFAIR OMOP — Reset Notebook

Run this notebook to wipe all the OMOP event tables and sequences back to a clean state.

**Use this when you want to start the ETL exercise from scratch.**

> ⚠️ This deletes all rows from PERSON, VISIT_OCCURRENCE, and all event tables in your database.
> The vocabulary tables (`concept`, `concept_relationship`, etc.) and the `import` schema are **not** touched.

After running this notebook, go back to the ETL exercise and run it from the top.

## Connection Settings

Replace `trainee_01` with your trainee number and fill in your password.

In [ ]:
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "trainee_01"   # replace with your trainee number
DB_USER     = "trainee_01"   # replace with your trainee number
DB_PASSWORD = "01"             # replace with your trainee number

print(f"Connecting to {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

con = psycopg2.connect(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
)
con.autocommit = True

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def execute(sql):
    with con.cursor() as cur:
        cur.execute(sql)

def query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

print("Connected to PostgreSQL.")

## Step 1 — Check current row counts

This shows how many rows are currently in each OMOP clinical table.

In [ ]:
query("""
SELECT 'person'               AS table_name, COUNT(*) AS row_count FROM omop.person
UNION ALL
SELECT 'observation_period',               COUNT(*) FROM omop.observation_period
UNION ALL
SELECT 'visit_occurrence',                 COUNT(*) FROM omop.visit_occurrence
UNION ALL
SELECT 'condition_occurrence',             COUNT(*) FROM omop.condition_occurrence
UNION ALL
SELECT 'measurement',                      COUNT(*) FROM omop.measurement
UNION ALL
SELECT 'observation',                      COUNT(*) FROM omop.observation
UNION ALL
SELECT 'procedure_occurrence',             COUNT(*) FROM omop.procedure_occurrence
UNION ALL
SELECT 'drug_exposure',                    COUNT(*) FROM omop.drug_exposure
ORDER BY table_name
""")

## Step 2 — Reset

Truncate all clinical tables (in reverse dependency order) and drop the sequences used for surrogate keys.

`TRUNCATE ... CASCADE` automatically clears any child tables that reference the truncated table.

In [ ]:
reset_sql = """
-- Wipe event tables first (they depend on person and visit)
TRUNCATE omop.procedure_occurrence  CASCADE;
TRUNCATE omop.drug_exposure         CASCADE;
TRUNCATE omop.measurement           CASCADE;
TRUNCATE omop.observation           CASCADE;
TRUNCATE omop.condition_occurrence  CASCADE;
TRUNCATE omop.visit_occurrence      CASCADE;
TRUNCATE omop.observation_period    CASCADE;
TRUNCATE omop.person                CASCADE;

-- Drop sequences so IDs restart from 1 on the next run
DROP SEQUENCE IF EXISTS condition_id_seq;
DROP SEQUENCE IF EXISTS observation_id_seq;
DROP SEQUENCE IF EXISTS measurement_id_seq;
DROP SEQUENCE IF EXISTS procedure_id_seq;
"""

execute(reset_sql)
print("Reset complete. All clinical tables are now empty.")

## Step 3 — Verify

All row counts below should be 0.

In [ ]:
query("""
SELECT 'person'               AS table_name, COUNT(*) AS row_count FROM omop.person
UNION ALL
SELECT 'observation_period',               COUNT(*) FROM omop.observation_period
UNION ALL
SELECT 'visit_occurrence',                 COUNT(*) FROM omop.visit_occurrence
UNION ALL
SELECT 'condition_occurrence',             COUNT(*) FROM omop.condition_occurrence
UNION ALL
SELECT 'measurement',                      COUNT(*) FROM omop.measurement
UNION ALL
SELECT 'observation',                      COUNT(*) FROM omop.observation
UNION ALL
SELECT 'procedure_occurrence',             COUNT(*) FROM omop.procedure_occurrence
UNION ALL
SELECT 'drug_exposure',                    COUNT(*) FROM omop.drug_exposure
ORDER BY table_name
""")

---
All done. Switch back to `hemafair_omop_etl_exercise.ipynb` and run it from the top.